<a href="https://colab.research.google.com/github/notgodfatherbms/AILAB/blob/main/1BF24CS056_WEEK8_FOL_Resolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# FIRST ORDER LOGIC RESOLUTION
# Case Study: Prove that John likes peanuts.

# ------------------------------------------------
# 1. REPRESENTATION OF KNOWLEDGE BASE
# ------------------------------------------------

# A literal is represented as:
# (negation, predicate, arguments)
# True = Negated, False = Positive
#
# Variables start with '?'
# Constants start with capital letters.

C1 = [(True, "Food", ("?x",)),
      (False, "Likes", ("John", "?x"))]

C2 = [(False, "Food", ("Apple",))]

C3 = [(False, "Food", ("Vegetables",))]

C4 = [(True, "Eats", ("?x", "?y")),
      (False, "Killed", ("?x",)),
      (False, "Food", ("?y",))]

C5 = [(False, "Eats", ("Anil", "Peanuts"))]

C6 = [(False, "Alive", ("Anil",))]

C7 = [(True, "Eats", ("Anil", "?x")),
      (False, "Eats", ("Harry", "?x"))]

C8 = [(True, "Alive", ("?x",)),
      (True, "Killed", ("?x",))]

C9 = [(False, "Killed", ("?x",)),
      (False, "Alive", ("?x",))]

# Negation of the query:
# Query = Likes(John, Peanuts)

C10 = [(True, "Likes", ("John", "Peanuts"))]

KB = [C1, C2, C3, C4, C5,
      C6, C7, C8, C9, C10]


# ------------------------------------------------
# 2. DISPLAY CLAUSES
# ------------------------------------------------

def format_literal(literal):
    neg, predicate, args = literal

    text = predicate + "(" + ", ".join(args) + ")"

    if neg:
        return "~" + text

    return text


def format_clause(clause):
    if not clause:
        return "EMPTY CLAUSE (□)"

    return " OR ".join(
        format_literal(lit) for lit in clause
    )


print("KNOWLEDGE BASE IN CNF\n")

for i, clause in enumerate(KB, 1):
    print("C" + str(i) + ":", format_clause(clause))


# ------------------------------------------------
# 3. UNIFICATION
# ------------------------------------------------

def is_variable(term):
    return term.startswith("?")


def substitute(term, substitution):

    while is_variable(term) and term in substitution:
        term = substitution[term]

    return term


def unify(term1, term2, substitution):

    term1 = substitute(term1, substitution)
    term2 = substitute(term2, substitution)

    if term1 == term2:
        return True

    if is_variable(term1):
        substitution[term1] = term2
        return True

    if is_variable(term2):
        substitution[term2] = term1
        return True

    return False


# ------------------------------------------------
# 4. APPLY SUBSTITUTION
# ------------------------------------------------

def apply_substitution(literal, substitution):

    neg, predicate, args = literal

    new_args = tuple(
        substitute(arg, substitution) for arg in args
    )

    return (neg, predicate, new_args)


# ------------------------------------------------
# 5. RESOLUTION
# ------------------------------------------------

def resolve(clause1, clause2):

    for i, lit1 in enumerate(clause1):

        for j, lit2 in enumerate(clause2):

            neg1, pred1, args1 = lit1
            neg2, pred2, args2 = lit2

            # Check complementary literals
            if pred1 != pred2 or neg1 == neg2:
                continue

            if len(args1) != len(args2):
                continue

            substitution = {}
            success = True

            # Perform unification
            for a, b in zip(args1, args2):

                if not unify(a, b, substitution):
                    success = False
                    break

            if success:

                # Remove resolved literals
                remaining = (
                    clause1[:i] + clause1[i+1:] +
                    clause2[:j] + clause2[j+1:]
                )

                # Apply substitution
                result = [
                    apply_substitution(lit, substitution)
                    for lit in remaining
                ]

                # Remove duplicate literals
                result = list(dict.fromkeys(result))

                return result

    return None


# ------------------------------------------------
# 6. PERFORM RESOLUTION FOR THE CASE STUDY
# ------------------------------------------------

print("\nRESOLUTION PROCESS\n")

proof = []

def perform_resolution(id1, id2, clause1, clause2, new_id):

    result = resolve(clause1, clause2)

    print("Resolving C" + str(id1) +
          " and C" + str(id2))

    if result is None:
        print("Resolution failed!")
        return None

    print("C" + str(new_id) + ":",
          format_clause(result))

    print()

    proof.append(
        (new_id, id1, id2, result)
    )

    return result


# Step 1: Resolve C6 and C8
# Alive(Anil) and ~Alive(x) OR ~Killed(x)

C11 = perform_resolution(6, 8, C6, C8, 11)


# Step 2: Resolve C4 and C5
# Anything eaten and not killed is food
# Anil eats peanuts

C12 = perform_resolution(4, 5, C4, C5, 12)


# Step 3: Resolve C12 and C11
# Killed(Anil) OR Food(Peanuts)
# ~Killed(Anil)

C13 = perform_resolution(12, 11, C12, C11, 13)


# Step 4: Resolve C1 and C13
# John likes all food
# Peanuts are food

C14 = perform_resolution(1, 13, C1, C13, 14)


# Step 5: Resolve C14 and C10
# Likes(John, Peanuts)
# ~Likes(John, Peanuts)

C15 = perform_resolution(14, 10, C14, C10, 15)


# ------------------------------------------------
# 7. DISPLAY FINAL RESULT
# ------------------------------------------------

print("FINAL RESULT")

if C15 == []:

    print("Empty clause generated: □")
    print("Contradiction obtained.")
    print("QUERY PROVED SUCCESSFULLY!")
    print("\nJohn likes peanuts.")

else:

    print("Query could not be proved.")


# ------------------------------------------------
# 8. DISPLAY RESOLUTION PROOF TREE
# ------------------------------------------------

print("\nRESOLUTION PROOF TREE\n")

print("""
                    C15 (□)
                   / \\
                C14 C10
                 |
        Likes(John, Peanuts)
                 |
                / \\
              C1 C13
                   |
             Food(Peanuts)
                   |
                  / \\
                C12 C11
                 | |
                 | ~Killed(Anil)
                 |
       Killed(Anil) OR Food(Peanuts)
                 |
                / \\
              C4 C5
""")

print("Derivation of C11:")
print("C6 + C8 = ~Killed(Anil)")


KNOWLEDGE BASE IN CNF

C1: ~Food(?x) OR Likes(John, ?x)
C2: Food(Apple)
C3: Food(Vegetables)
C4: ~Eats(?x, ?y) OR Killed(?x) OR Food(?y)
C5: Eats(Anil, Peanuts)
C6: Alive(Anil)
C7: ~Eats(Anil, ?x) OR Eats(Harry, ?x)
C8: ~Alive(?x) OR ~Killed(?x)
C9: Killed(?x) OR Alive(?x)
C10: ~Likes(John, Peanuts)

RESOLUTION PROCESS

Resolving C6 and C8
C11: ~Killed(Anil)

Resolving C4 and C5
C12: Killed(Anil) OR Food(Peanuts)

Resolving C12 and C11
C13: Food(Peanuts)

Resolving C1 and C13
C14: Likes(John, Peanuts)

Resolving C14 and C10
C15: EMPTY CLAUSE (□)

FINAL RESULT
Empty clause generated: □
Contradiction obtained.
QUERY PROVED SUCCESSFULLY!

John likes peanuts.

RESOLUTION PROOF TREE


                    C15 (□)
                   / \
                C14 C10
                 |
        Likes(John, Peanuts)
                 |
                / \
              C1 C13
                   |
             Food(Peanuts)
                   |
                  / \
                C12 C11
             

### Dynamic FOL Resolution Prover
Below is an automated implementation of the resolution algorithm. It automatically searches for pairs of clauses that can resolve, applies unification, and tracks the ancestry of each resolved clause to print the full proof steps once the empty clause (□) is found.

In [2]:
def standardize_variables(clause, suffix):
    """
    Renames variables in a clause by adding a unique suffix to avoid
    variable name collisions during unification of different clauses.
    """
    new_clause = []
    for neg, predicate, args in clause:
        new_args = tuple(
            (arg + suffix) if is_variable(arg) else arg
            for arg in args
        )
        new_clause.append((neg, predicate, new_args))
    return new_clause

def clauses_are_equal(c1, c2):
    """Checks if two clauses are equivalent (set comparison)."""
    return set(c1) == set(c2)

def automate_resolution(initial_kb):
    """
    Automatically finds a proof using a breadth-first search strategy.
    Tracks parents to construct and display the proof path.
    """
    # Create working set of clauses with metadata: (clause_id, clause, parent_ids_tuple)
    clauses = []
    for idx, c in enumerate(initial_kb, 1):
        clauses.append((idx, c, ()))

    next_id = len(initial_kb) + 1
    checked_pairs = set()

    print("--- STARTING AUTOMATED RESOLUTION SEARCH ---\n")

    step = 0
    max_steps = 1000  # Avoid infinite loop in case of unprovable queries

    while step < len(clauses):
        id1, clause1, parents1 = clauses[step]

        for id2, clause2, parents2 in clauses[:step]:
            pair = tuple(sorted((id1, id2)))
            if pair in checked_pairs:
                continue
            checked_pairs.add(pair)

            # Standardize variables apart to avoid namespace clashes
            c1_std = standardize_variables(clause1, f"_c{id1}")
            c2_std = standardize_variables(clause2, f"_c{id2}")

            resolved = resolve(c1_std, c2_std)

            if resolved is not None:
                # Check if this clause (or an equivalent one) already exists
                duplicate = False
                for existing_id, existing_clause, _ in clauses:
                    if clauses_are_equal(resolved, existing_clause):
                        duplicate = True
                        break

                if not duplicate:
                    new_clause_entry = (next_id, resolved, (id1, id2))
                    clauses.append(new_clause_entry)
                    print(f"C{next_id} obtained by resolving C{id1} and C{id2}: {format_clause(resolved)}")

                    # If empty clause is generated, we are done!
                    if not resolved:
                        print(f"\nEmpty clause (□) generated as C{next_id}!")
                        print("QUERY PROVED SUCCESSFULLY!\n")
                        reconstruct_proof(clauses, new_clause_entry)
                        return True

                    next_id += 1
        step += 1

    print("Resolution search completed. Query could not be proved.")
    return False

def reconstruct_proof(all_clauses, final_entry):
    """Traverses backwards from the empty clause to print only the relevant proof steps."""
    clause_dict = {item[0]: item for item in all_clauses}
    proof_steps = []

    def collect(entry):
        cid, clause, parents = entry
        if parents:
            p1, p2 = parents
            collect(clause_dict[p1])
            collect(clause_dict[p2])
            proof_steps.append(entry)

    collect(final_entry)

    # Remove duplicate steps in the path reconstruction
    seen = set()
    unique_steps = []
    for step in proof_steps:
        if step[0] not in seen:
            seen.add(step[0])
            unique_steps.append(step)

    print("--- RECONSTRUCTED PROOF PATH ---")
    for cid, clause, parents in unique_steps:
        p1, p2 = parents
        print(f"Step: Resolve C{p1} and C{p2} -> C{cid}: {format_clause(clause)}")

# Run the automated resolution engine
automate_resolution(KB)

--- STARTING AUTOMATED RESOLUTION SEARCH ---

C11 obtained by resolving C2 and C1: Likes(John, Apple)
C12 obtained by resolving C3 and C1: Likes(John, Vegetables)
C13 obtained by resolving C4 and C1: ~Eats(?x_c4, ?x_c1) OR Killed(?x_c4) OR Likes(John, ?x_c1)
C14 obtained by resolving C5 and C4: Killed(Anil) OR Food(Peanuts)
C15 obtained by resolving C7 and C4: ~Eats(Anil, ?y_c4) OR Killed(Harry) OR Food(?y_c4)
C16 obtained by resolving C7 and C5: Eats(Harry, Peanuts)
C17 obtained by resolving C8 and C4: ~Alive(?x_c4) OR ~Eats(?x_c4, ?y_c4) OR Food(?y_c4)
C18 obtained by resolving C8 and C6: ~Killed(Anil)
C19 obtained by resolving C9 and C8: Alive(?x_c8) OR ~Alive(?x_c8)
C20 obtained by resolving C10 and C1: ~Food(Peanuts)
C21 obtained by resolving C13 and C5: Killed(Anil) OR Likes(John, Peanuts)
C22 obtained by resolving C13 and C7: Killed(Harry) OR Likes(John, ?x_c7) OR ~Eats(Anil, ?x_c7)
C23 obtained by resolving C13 and C8: ~Eats(?x_c8, ?x_c1_c13) OR Likes(John, ?x_c1_c13) OR ~Alive

True

In [3]:
class Node:
    def __init__(self, value, rule="", children=None):
        self.value = value
        self.rule = rule
        self.children = children if children else []


def display_tree(node, prefix="", is_last=True):
    connector = "└── " if is_last else "├── "
    print(prefix + connector + node.value)

    if node.rule:
        print(prefix + ("    " if is_last else "│   ") + node.rule)

    for i, child in enumerate(node.children):
        display_tree(
            child,
            prefix + ("    " if is_last else "│   "),
            i == len(node.children) - 1
        )


print("FIRST ORDER LOGIC - RESOLUTION")
print("--------------------------------")

print("\n1. KNOWLEDGE BASE")

clauses = [
    "C1: ~food(x) OR likes(John,x)",
    "C2: food(Apple)",
    "C3: food(Vegetables)",
    "C4: ~eats(x,y) OR killed(x) OR food(y)",
    "C5: eats(Anil,Peanuts)",
    "C6: alive(Anil)",
    "C7: ~alive(x) OR ~killed(x)",
    "C8: killed(x) OR alive(x)"
]

for c in clauses:
    print(c)

print("\n2. NEGATE THE QUERY")

print("Query: likes(John,Peanuts)")
print("Negated Query: ~likes(John,Peanuts)")

print("\n3. RESOLUTION STEPS")

print("\nStep 1:")
print("C1: ~food(x) OR likes(John,x)")
print("C9: ~likes(John,Peanuts)")
print("Substitution: x = Peanuts")
print("Resolvent R1: ~food(Peanuts)")

print("\nStep 2:")
print("C4: ~eats(x,y) OR killed(x) OR food(y)")
print("R1: ~food(Peanuts)")
print("Substitution: y = Peanuts")
print("Resolvent R2: ~eats(x,Peanuts) OR killed(x)")

print("\nStep 3:")
print("C5: eats(Anil,Peanuts)")
print("R2: ~eats(x,Peanuts) OR killed(x)")
print("Substitution: x = Anil")
print("Resolvent R3: killed(Anil)")

print("\nStep 4:")
print("C6: alive(Anil)")
print("C7: ~alive(x) OR ~killed(x)")
print("Substitution: x = Anil")
print("Resolvent R4: ~killed(Anil)")

print("\nStep 5:")
print("R3: killed(Anil)")
print("R4: ~killed(Anil)")
print("Resolvent: {} (Empty Clause)")

print("\n4. GENERATING PROOF TREE")

# Leaf clauses
goal = Node("~likes(John,Peanuts)")
c1 = Node("~food(x) OR likes(John,x)")
c4 = Node("~eats(x,y) OR killed(x) OR food(y)")
c5 = Node("eats(Anil,Peanuts)")
c6 = Node("alive(Anil)")
c7 = Node("~alive(x) OR ~killed(x)")

# Resolution R1
r1 = Node(
    "~food(Peanuts)",
    "Unification: x = Peanuts",
    [goal, c1]
)

# Resolution R2
r2 = Node(
    "~eats(x,Peanuts) OR killed(x)",
    "Unification: y = Peanuts",
    [r1, c4]
)

# Resolution R3
r3 = Node(
    "killed(Anil)",
    "Unification: x = Anil",
    [r2, c5]
)

# Resolution R4
r4 = Node(
    "~killed(Anil)",
    "Unification: x = Anil",
    [c6, c7]
)

# Final empty clause
root = Node(
    "{} EMPTY CLAUSE",
    "Contradiction: killed(Anil) AND ~killed(Anil)",
    [r3, r4]
)

display_tree(root)

print("\n5. FINAL RESULT")
print("Empty clause obtained: {}")
print("Contradiction found!")
print("Therefore, John likes peanuts is PROVED.")

FIRST ORDER LOGIC - RESOLUTION
--------------------------------

1. KNOWLEDGE BASE
C1: ~food(x) OR likes(John,x)
C2: food(Apple)
C3: food(Vegetables)
C4: ~eats(x,y) OR killed(x) OR food(y)
C5: eats(Anil,Peanuts)
C6: alive(Anil)
C7: ~alive(x) OR ~killed(x)
C8: killed(x) OR alive(x)

2. NEGATE THE QUERY
Query: likes(John,Peanuts)
Negated Query: ~likes(John,Peanuts)

3. RESOLUTION STEPS

Step 1:
C1: ~food(x) OR likes(John,x)
C9: ~likes(John,Peanuts)
Substitution: x = Peanuts
Resolvent R1: ~food(Peanuts)

Step 2:
C4: ~eats(x,y) OR killed(x) OR food(y)
R1: ~food(Peanuts)
Substitution: y = Peanuts
Resolvent R2: ~eats(x,Peanuts) OR killed(x)

Step 3:
C5: eats(Anil,Peanuts)
R2: ~eats(x,Peanuts) OR killed(x)
Substitution: x = Anil
Resolvent R3: killed(Anil)

Step 4:
C6: alive(Anil)
C7: ~alive(x) OR ~killed(x)
Substitution: x = Anil
Resolvent R4: ~killed(Anil)

Step 5:
R3: killed(Anil)
R4: ~killed(Anil)
Resolvent: {} (Empty Clause)

4. GENERATING PROOF TREE
└── {} EMPTY CLAUSE
    Contradiction: